# 01 · Contextual Retrieval job (one-off, ~2–2.5 h on a T4)

Runs **A4 CONTEXTUALIZE** from `docs/02-architecture.md`: **Qwen3.5-4B** on vLLM writes one context line per chunk,
**one prompt per document** (document sent once, with inline `[[c1]]…[[cN]]` markers).

- Input: `artifacts/chunks.parquet` (committed in the repo, so chunk ids are identical everywhere): 8,627 chunks.
- Output: `contexts_<model>.jsonl` `{chunk_id, context, model}`. It's written to Google Drive if mounted, so the job is
  **resumable** after a Colab disconnect: just re-run all cells and finished chunks are skipped. The file name carries
  the model, so rows from the earlier Qwen2.5-3B run are never mixed in.
- vLLM runs in its **own venv as a background subprocess**: it pins its own torch, and its GPU memory is released on exit.
- **Run the smoke test first** (2 documents). Qwen3.5 is new (hybrid Gated-DeltaNet attention, bf16 weights run in
  fp16 on a T4); if the smoke test fails, switch `MODEL` to the fallback `Qwen/Qwen3-4B-Instruct-2507`.

When it's done, download the file and commit it as `artifacts/contexts.jsonl`. The demo notebook
(`02_demo_and_eval.ipynb`) never re-runs this; it also finds the file on Drive by itself.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import os, subprocess, json, time
REPO = "https://github.com/nrusnac/DP_task_RAG.git"
BRANCH = "agent-improvements"
if not os.path.exists("/content/DP_task_RAG"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/DP_task_RAG"], check=True)
os.chdir("/content/DP_task_RAG")
# sync to the latest commit of BRANCH; outputs of earlier runs in this clone are discarded (they are saved separately)
subprocess.run(["git", "fetch", "-q", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
!git log --oneline -1
!pip -q install -e .

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"   # fallback if the smoke test fails: "Qwen/Qwen3-4B-Instruct-2507"
SLUG = MODEL.split("/")[-1].lower()
# Optional but recommended: write the output to Drive so a disconnect doesn't lose progress
OUT_DIR = "/content/DP_task_RAG/artifacts"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/dp_task_rag"
    os.makedirs(OUT_DIR, exist_ok=True)
except Exception as e:
    print("Drive not mounted, writing locally:", e)
OUT = f"{OUT_DIR}/contexts_{SLUG}.jsonl"
print("model ->", MODEL, "| output ->", OUT)

In [ ]:
!python -m hrag.ingest.contextualize prepare
N_CHUNKS = sum(len(j["chunk_ids"]) for j in json.load(open("data/index/ctx_input.json"))["jobs"])
print("chunks to contextualize:", N_CHUNKS)

In [ ]:
# vLLM in an isolated venv (~2 min). Qwen3.5 needs vLLM >= 0.17.
if not os.path.exists("/content/vllm_env/bin/python"):
    !pip -q install uv
    !uv venv -q /content/vllm_env --python 3.12
    !uv pip install -q --python /content/vllm_env/bin/python vllm
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

**Smoke test** (~3–5 min, mostly model loading): 2 documents into a scratch file. Pass = no error, `coverage` ≥ 0.95
and the sample lines read like `[c1] <one sentence about the matter and the chunk>`.
- Kernel/CUDA error or garbage text → set `MODEL = "Qwen/Qwen3-4B-Instruct-2507"` above, re-run that cell and this one.
- Out of GPU memory → add `--enforce-eager` to this command and to the launch cell.

In [ ]:
SMOKE = "/content/ctx_smoke.jsonl"
if os.path.exists(SMOKE):
    os.remove(SMOKE)
!/content/vllm_env/bin/python scripts/contextualize_vllm.py data/index/ctx_input.json {SMOKE} --model {MODEL} --limit 2 2>&1 | grep -vE "^(INFO|DEBUG)|it/s\]" | tail -30

In [ ]:
# Launch in the background: this cell returns immediately and the job keeps running while the runtime is alive
log = open("/content/ctx.log", "a")
proc = subprocess.Popen(["/content/vllm_env/bin/python", "scripts/contextualize_vllm.py", "data/index/ctx_input.json", OUT,
                         "--model", MODEL],
                        stdout=log, stderr=subprocess.STDOUT, cwd="/content/DP_task_RAG")
print("started pid", proc.pid, "| log: /content/ctx.log")

Re-run the cell below any time to check progress (the first ~3 min are model loading and compilation).

In [ ]:
status = "running" if proc.poll() is None else f"exited with code {proc.returncode}"
print("status:", status)
!grep -E "jobs|Error|error" /content/ctx.log | tail -4
n = sum(1 for _ in open(OUT)) if os.path.exists(OUT) else 0
print(f"chunks with a context row: {n} / {N_CHUNKS}")
if proc.poll() not in (None, 0):
    !tail -40 /content/ctx.log

In [ ]:
# When finished: download the file and commit it as artifacts/contexts.jsonl
from google.colab import files
files.download(OUT)